# Round 2: PCA on the 1000G superpopulation, OADP projection, robust gate

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run, repo_root
from aoucov.batch import run_script
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = cfg.params.get("threads", 4)

In [ ]:
from ancestry import mahalanobis, fit_robust_gate, gate_ellipse
from stage_helpers import hexbin_safe, peak_enrichment
PCA_OUT = single_target(cfg, "round2_pca")
PCA_DIR = os.path.dirname(PCA_OUT)
require_ready(cfg, "round2_pca")
rep_pca = Report(cfg, "round2_pca")

## PCA and projection (bigsnpr)

In [ ]:
if needs_run(PCA_OUT, FORCE):
    with record(cfg, "round2_pca", PCA_OUT, tools={"bigsnpr": "bed_projectPCA"}):
        run_script(cfg, "round2_pca.R", dict(
            REF_BFILE=cfg.input("ref_bfile"), REF_LABELS=cfg.input("ref_labels"),
            TRAIN_POPS=",".join(cfg.param("r2_train_pops")), NEW_BED=single_target(cfg, "hm3_panel"),
            K=cfg.param("r2_n_pcs_fit"), LD_REGIONS=cfg.input("ld_regions"), OUT_DIR=PCA_DIR, NCORES=THREADS))

In [ ]:
meta = json.load(open(f"{PCA_DIR}/round2_meta.json"))
scores = pd.read_csv(PCA_OUT, sep="\t", dtype={"id": str})
simple = pd.read_csv(f"{PCA_DIR}/simple_projection.tsv", sep="\t", dtype={"id": str})
load = pd.read_csv(f"{PCA_DIR}/loadings.tsv", sep="\t")
lrldr = pd.read_csv(f"{PCA_DIR}/lrldr.tsv", sep="\t")
PCS = [c for c in scores if c.startswith("PC")]
pcs_v = [f"V{k}" for k in range(1, len(PCS) + 1)]
load = load.rename(columns=dict(zip(pcs_v, PCS)))
ref_s, coh = scores[scores["set"] == "reference"], scores[scores["set"] == "cohort"]

rep_pca.note(train_pops=",".join(cfg.param("r2_train_pops")), n_train=meta["n_train"],
             n_variants_used=meta["n_variants_used"], n_listed_ld_excluded=meta["n_listed_ld_excluded"])
rep_pca.check("fit on reference founders only (no relatives in the fit)", True, f"{meta['n_train']} founders")
rep_pca.check("long-range LD detected and removed (bed_autoSVD)", True,
              "; ".join(f"chr{r.Chr}:{r.Start/1e6:.0f}-{r.Stop/1e6:.0f} Mb" for r in lrldr.itertuples()) or "none found")

In [ ]:
r2_pop = {pc: ref_s.groupby("pop")[pc].transform("mean").var() / ref_s[pc].var() for pc in PCS}
diag = pd.DataFrame({"pc": PCS, "r2_population": [round(r2_pop[p], 3) for p in PCS],
                     "loading_peak": peak_enrichment(load, PCS).round(1).values,
                     "oadp_over_simple_sd": (coh[PCS].std().values / simple[PCS].std().values).round(2)})
rep_pca.table(diag, "pc_diagnostics", caption="pick K: structure (r2_population) without a loading peak")
diag

In [ ]:
fig, axes = plt.subplots(len(PCS[:6]), 1, figsize=(12, 1.6 * len(PCS[:6])), sharex=True)
for ax, pc in zip(axes, PCS):
    ax.scatter(np.arange(len(load)), load[pc].abs(), s=0.5, c=load["chr"] % 2, cmap="coolwarm", rasterized=True)
    ax.set_ylabel(pc)
axes[-1].set_xlabel("variant (genome order)")
rep_pca.fig(fig, "loadings"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (i, j) in zip(axes, [(0, 1), (2, 3), (4, 5)]):
    for p, d in ref_s.groupby("pop"):
        ax.scatter(d[PCS[i]], d[PCS[j]], s=8, label=p)
    ax.set(xlabel=PCS[i], ylabel=PCS[j])
axes[0].legend(fontsize=8)
rep_pca.fig(fig, "reference_scores", "1000G training samples"); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, pc in zip(axes, PCS[:4]):
    ax.hist2d(simple[pc], coh[pc], bins=80, cmin=21, cmap="Blues")
    lim = np.abs(coh[pc]).quantile(0.995)
    ax.plot([-lim, lim], [-lim, lim], "k:", lw=0.8)
    ax.set(xlabel=f"{pc} simple", ylabel=f"{pc} OADP")
rep_pca.fig(fig, "simple_vs_oadp", "simple projection shrinks toward 0"); plt.show()
rep_pca.save()

## Gate (set `r2_k` and `r2_q` in the run config from the table above)

In [ ]:
STAGE = "ancestry_round2"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
K, Q, ANCHOR = cfg.param("r2_k"), cfg.param("r2_q"), cfg.param("r2_anchor_pops")
PCK = PCS[:K]
A = ref_s[ref_s["pop"].isin(ANCHOR)][PCK].to_numpy()
g = fit_robust_gate(A, Q)
d = mahalanobis(coh[PCK].to_numpy(), g["mu"], g["C"])
keep = d <= g["radius"]
d_simple = mahalanobis(simple[PCK].to_numpy(), g["mu"], g["C"])

rep.note(k=K, q=Q, anchor=",".join(ANCHOR), n_anchor=len(A), anchor_outliers=g["anchor_outliers"],
         radius=round(g["radius"], 3))
rep.count("n_round1", len(coh))
print("kept", rep.count("n_kept", keep.sum()))
rep.count("n_kept_if_simple_projection", (d_simple <= g["radius"]).sum())
rep.check("anchor inside the gate", (g["ref_d"] <= g["radius"]).mean() >= 0.9,
          f"{(g['ref_d'] <= g['radius']).mean():.1%}")
rep.check("every PC used carries population structure", all(diag["r2_population"][:K] > 0.02))
rep.check("no PC used is driven by one region", all(diag["loading_peak"][:K] < 10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (i, j) in zip(axes, [(0, 1), (2, 3)]):
    if j >= K:
        ax.axis("off"); continue
    hexbin_safe(ax, coh[PCS[i]], coh[PCS[j]])
    for p, dd in ref_s[ref_s["pop"].isin(cfg.param("r2_train_pops"))].groupby("pop"):
        ax.scatter(dd[PCS[i]], dd[PCS[j]], s=8, marker="x", label=p)
    gate_ellipse(ax, g["mu"], g["C"], i, j, g["radius"], edgecolor="tab:red", lw=1.5)
    ax.set(xlabel=PCS[i], ylabel=PCS[j])
axes[0].legend(fontsize=8)
rep.fig(fig, "gate_pcs"); plt.show()

qs = np.array([0.9, 0.99, 0.999, 0.9999, 0.99999])
from scipy.stats import chi2
kept_q = [int((d <= np.sqrt(chi2.ppf(q, K))).sum()) for q in qs]
rep.table(pd.DataFrame({"q": qs, "kept": kept_q}), "kept_vs_q", counts=["kept"])

## Write

In [ ]:
if needs_run(OUT, FORCE):
    with record(cfg, STAGE, OUT) as meta:
        get_store(OUT).write_text(OUT, "\n".join(coh.loc[keep, "id"]) + "\n")
        meta.update(n_kept=int(keep.sum()), k=K, q=Q)
rep.save()